# 💉 Vaccine Cold Chain Last-Mile Delivery — GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalitsridatta/SuperNova/blob/main/vaccine_cold_chain_training.ipynb)

Trains a **Qwen3-0.6B** LLM agent using **GRPO (TRL)** to manage vaccine delivery under cold chain constraints.

The agent uses 3 tools: `go_to_stop()`, `restock_ice()`, `wait()` to deliver vaccines to 4 clinics while keeping temperature between **2–8°C**.

**Environment:** https://huggingface.co/spaces/Lsd45/vaccine-cold-chain

In [ ]:
# Install dependencies
!pip install -q trl>=0.12.0 transformers datasets accelerate peft requests matplotlib

In [ ]:
# Verify environment is reachable
import requests

ENV_URL = 'https://lsd45-vaccine-cold-chain.hf.space'

r = requests.get(f'{ENV_URL}/health', timeout=30)
print('Health:', r.json())

obs = requests.post(f'{ENV_URL}/reset', timeout=30).json()
print('Ice:', obs['ice_remaining'], '| Temp:', obs['current_temperature'], '°C')
print('Stops:', [s['stop_id'] for s in obs['stops_remaining']])
print('Emergency:', obs.get('emergency_stop'))

In [ ]:
import time
import requests as req
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ENV_URL = 'https://lsd45-vaccine-cold-chain.hf.space'

# Resilient session — auto-retries if space goes to sleep
def make_session():
    s = req.Session()
    retry = Retry(total=5, backoff_factor=2, status_forcelist=[500, 502, 503, 504])
    s.mount('https://', HTTPAdapter(max_retries=retry))
    return s

_session = make_session()

def resilient_post(url, **kwargs):
    kwargs.setdefault('timeout', 60)
    for attempt in range(3):
        try:
            r = _session.post(url, **kwargs)
            r.raise_for_status()
            return r
        except Exception as e:
            if attempt < 2:
                print(f'Retrying ({e})...')
                time.sleep(10)
            else:
                raise

SYSTEM_PROMPT = (
    'You are an AI agent managing vaccine last-mile delivery under cold chain constraints.\n\n'
    'Your goal: deliver vaccines to all stops while keeping the temperature between 2-8 degrees C.\n\n'
    'Key rules:\n'
    '- Ice depletes over time and faster when traveling.\n'
    '- If ice drops below 30, restock immediately or vaccines will spoil.\n'
    '- Temperature outside safe range for 3+ consecutive steps = mission failure.\n'
    '- Deliver emergency stops as soon as possible for bonus reward.\n'
    '- Avoid unnecessary restocking when ice is already high.\n\n'
    'Available tools:\n'
    '- go_to_stop(stop_id): Drive to a delivery stop and deliver vaccines.\n'
    '- restock_ice(): Return to depot to refill ice (costs 1 hour).\n'
    '- wait(): Stay in place for one step (avoid unless necessary).\n\n'
    'Think step by step before acting.'
)


class VaccineDeliveryToolEnv:

    def __init__(self):
        self.reward = 0.0
        self.done = False
        self._total_reward = 0.0

    def reset(self, **kwargs):
        self.reward = 0.0
        self.done = False
        self._total_reward = 0.0
        obs = resilient_post(f'{ENV_URL}/reset').json()
        return self._fmt(obs)

    def go_to_stop(self, stop_id: str) -> str:
        """
        Drive to a delivery stop and deliver vaccines.

        Args:
            stop_id: The stop ID to deliver to. Valid values: clinic_a, clinic_b, village_c, health_post_d.

        Returns:
            Updated environment status.
        """
        return self._step({'action_type': 'go_to_stop', 'stop_id': stop_id})

    def restock_ice(self) -> str:
        """
        Return to depot to refill ice. Use when ice is below 30.

        Returns:
            Updated environment status.
        """
        return self._step({'action_type': 'restock_ice'})

    def wait(self) -> str:
        """
        Stay in place for one step. Ice still depletes. Avoid unless necessary.

        Returns:
            Updated environment status.
        """
        return self._step({'action_type': 'wait'})

    def _step(self, action):
        if self.done:
            raise ValueError('Episode is over.')
        obs = resilient_post(f'{ENV_URL}/step', json=action).json()
        self._total_reward += obs.get('last_reward', 0.0)
        self.done = obs.get('done', False)
        if self.done:
            self.reward = max(0.0, min(1.0, (self._total_reward + 2.0) / 4.0))
            raise ValueError(f"Episode complete: {obs.get('info', '')}")
        return self._fmt(obs)

    def _fmt(self, obs):
        stops = obs.get('stops_remaining', [])
        stops_text = ', '.join(f"{s['stop_id']}(p{s['priority']})" for s in stops) or 'None'
        return (
            f"Ice: {obs.get('ice_remaining')}/100 | "
            f"Temp: {obs.get('current_temperature')}C (safe:2-8C) | "
            f"Outside: {obs.get('outside_temperature')}C | "
            f"Time: {obs.get('time_of_day')}h | "
            f"Excursion: {obs.get('temperature_excursion_steps')}/3 | "
            f"Emergency: {obs.get('emergency_stop', 'None')} | "
            f"Stops left: {stops_text} | "
            f"Info: {obs.get('info', '')}"
        )


def reward_func(environments, **kwargs):
    return [env.reward for env in environments]


print('Environment wrapper ready.')

In [ ]:
from datasets import Dataset

NUM_EPISODES = 300

prompt = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'A new vaccine delivery mission has started. Use your tools to complete all deliveries while keeping vaccines cold.'}
]

dataset = Dataset.from_dict({'prompt': [prompt] * NUM_EPISODES})
print(f'Dataset: {len(dataset)} episodes')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

MODEL = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = './vaccine_agent_trained'

config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=4,
    gradient_accumulation_steps=8,
    max_completion_length=512,
    per_device_train_batch_size=1,
    chat_template_kwargs={'enable_thinking': False},
    log_completions=True,
    logging_steps=5,
    save_steps=50,
    num_train_epochs=1,
)

trainer = GRPOTrainer(
    model=MODEL,
    train_dataset=dataset,
    reward_funcs=reward_func,
    args=config,
    environment_factory=VaccineDeliveryToolEnv,
)

print(f'Starting GRPO training with {MODEL}...')
trainer.train()

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
reward_key = next((k for k in ['reward', 'train/reward'] if any(k in l for l in log_history)), None)
rewards = [(l['step'], l[reward_key]) for l in log_history if reward_key and reward_key in l]
losses  = [(l['step'], l['loss']) for l in log_history if 'loss' in l]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if rewards:
    steps, vals = zip(*rewards)
    axes[0].plot(steps, vals, marker='o', linewidth=2, color='steelblue', markersize=4)
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Reward (normalized 0-1)')
    axes[0].set_title('GRPO Training Reward — Vaccine Cold Chain')
    axes[0].grid(True, alpha=0.3)

if losses:
    steps, vals = zip(*losses)
    axes[1].plot(steps, vals, linewidth=2, color='tomato')
    axes[1].set_xlabel('Training Step')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Training Loss')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print('Saved reward_curve.png')

In [ ]:
trainer.save_model(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

# Optional: push to HF Hub
# trainer.push_to_hub('YOUR_USERNAME/vaccine-cold-chain-agent')